# Детектор AI-текста на основе трансформера

## 1. Постановка задачи

Рассматривается задача бинарной классификации текста:

- класс `0` соответствует тексту, написанному человеком;
- класс `1` соответствует тексту, сгенерированному моделью.

## 2. Почему в работе используется трансформер

- учитывает контекст по всей последовательности;
- способен выявлять дальние зависимости между словами;
- хорошо подходит для анализа стиля и структуры текста;
- является современной базовой архитектурой для задач обработки естественного языка.

## 3. План работы

В ноутбуке последовательно выполняются следующие этапы:

1. Импорт библиотек.
2. Задание конфигурации эксперимента.
3. Реализация токенизатора и датасета.
4. Реализация трансформерной модели.
5. Реализация PyTorch Lightning модуля.
6. Реализация полного цикла обучения с прогресс-барами.
7. Загрузка данных и обучение.
8. Оценка на тестовой выборке.
9. Анализ по источникам данных.
10. Демонстрация предсказаний на отдельных примерах.
11. Итоговые выводы.

## 4. Импорт библиотек

In [1]:
from __future__ import annotations

import json
import math
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer
from tokenizers import decoders, models, normalizers, pre_tokenizers, trainers
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from dataset_loader import DatasetConfig, Sample, load_combined_dataset


/Users/ep/Documents/Masters/project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Конфигурация источников

In [2]:
ALL_SOURCES = [
    "raid",
    "ai_pile",
    "gpt_wiki",
    "human_vs_ai",
    "ai_human_mixed",
    "hc3",
    "coat",
    "ruatd",
    "daigt_proper",
    "daigt_v2",
    "m_daigt",
    "gsingh_train",
]

artifact_dir = Path("checkpoints/student_transformer_detector")
test_ratio = 0.15

source_paths = {
    # При необходимости можно указать локальные пути.
    # "daigt_proper": "/абсолютный/путь/к/train_v2_drcat_02.csv",
    # "m_daigt": "/абсолютный/путь/к/каталогу_m_daigt",
}

data_config = DatasetConfig(
    sources=ALL_SOURCES,
    source_paths=source_paths,
    max_per_source=100_000,
    max_total=200_000,
    min_text_length=35,
    max_text_length=3_000,
    balance_labels=True,
    balance_sources=False,
    seed=42,
    cache_dir=".cache/datasets",
    auto_download_kaggle=True,
    cache_sources=True,
)

live_examples = [
    ("Ну блин, опять забыл зонт и промок как собака.", "Human"),
    ("Just grabbed tacos, best decision I've made all week lol", "Human"),
    ("spent 3 hours debugging only to find a missing comma", "Human"),
    (
        "The implementation of machine learning algorithms in healthcare has demonstrated significant potential for improving diagnostic accuracy.",
        "AI",
    ),
    (
        "In conclusion, the comprehensive analysis reveals a multifaceted landscape that necessitates careful consideration.",
        "AI",
    ),
    (
        "Furthermore, it is important to note that the integration of these systems requires a holistic understanding of the underlying frameworks.",
        "AI",
    ),
]

print("Каталог артефактов:", artifact_dir)
print("Количество источников:", len(data_config.sources))


Каталог артефактов: checkpoints/student_transformer_detector
Количество источников: 12


## 6. Вспомогательные функции

In [3]:
def format_seconds(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def configure_torch_runtime() -> None:
    if not torch.cuda.is_available():
        return
    torch.set_float32_matmul_precision("high")
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = True


def resolve_num_workers(num_workers: int) -> int:
    if num_workers >= 0:
        return num_workers
    cpu_count = os.cpu_count() or 2
    return max(1, min(11, cpu_count - 1))


In [4]:
def build_stratify_labels(labels: list[int]) -> list[int] | None:
    if len(set(labels)) < 2:
        return None
    counts = Counter(labels)
    if min(counts.values()) < 2:
        return None
    return labels


def split_train_val(
    texts: list[str],
    labels: list[int],
    val_split: float,
    seed: int,
) -> tuple[list[str], list[str], list[int], list[int]]:
    stratify = build_stratify_labels(labels)
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts,
        labels,
        test_size=val_split,
        random_state=seed,
        stratify=stratify,
    )
    return train_texts, val_texts, train_labels, val_labels


def split_loaded_data(result, test_ratio: float, seed: int):
    indices = list(range(len(result.texts)))
    groups = [f"{result.samples[i].source_dataset}:{result.labels[i]}" for i in indices]
    group_counts = Counter(groups)

    stratify = None
    if group_counts and min(group_counts.values()) >= 2:
        stratify = groups
    else:
        label_counts = Counter(result.labels)
        if label_counts and min(label_counts.values()) >= 2:
            stratify = result.labels

    train_idx, test_idx = train_test_split(
        indices,
        test_size=test_ratio,
        random_state=seed,
        stratify=stratify,
    )

    train_texts = [result.texts[i] for i in train_idx]
    train_labels = [result.labels[i] for i in train_idx]
    test_texts = [result.texts[i] for i in test_idx]
    test_labels = [result.labels[i] for i in test_idx]
    test_samples = [result.samples[i] for i in test_idx]
    return train_texts, train_labels, test_texts, test_labels, test_samples


## 7. Токенизация и подготовка данных

Для представления текста используется байтовый BPE-токенизатор.

In [5]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]"]


class NotebookBPETokenizer:
    """Простой BPE-токенизатор"""

    def __init__(self, backend_tokenizer, max_len: int = 512):
        self.backend = backend_tokenizer
        self.max_len = max_len
        self.backend.enable_truncation(max_length=max_len)
        self._pad_id = self.backend.token_to_id("[PAD]")
        self._unk_id = self.backend.token_to_id("[UNK]")

    @classmethod
    def from_texts(
        cls,
        texts: list[str],
        max_vocab: int = 30_000,
        max_len: int = 512,
        min_frequency: int = 2,
    ):
        backend = Tokenizer(models.BPE(unk_token="[UNK]"))
        backend.normalizer = normalizers.NFC()
        backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        backend.decoder = decoders.ByteLevel()

        trainer = trainers.BpeTrainer(
            vocab_size=max_vocab,
            min_frequency=min_frequency,
            special_tokens=SPECIAL_TOKENS,
            show_progress=False,
        )

        def text_iterator():
            for text in tqdm(texts, desc="Обучение токенизатора", unit="text", leave=False):
                yield text

        backend.train_from_iterator(text_iterator(), trainer=trainer, length=len(texts))
        return cls(backend, max_len=max_len)

    def encode(self, text: str) -> list[int]:
        ids = self.backend.encode(text).ids[: self.max_len]
        if not ids:
            ids = [self.unk_idx]
        return ids

    @property
    def vocab_size(self) -> int:
        return self.backend.get_vocab_size()

    @property
    def pad_idx(self) -> int:
        return int(self._pad_id)

    @property
    def unk_idx(self) -> int:
        return int(self._unk_id)

    def save(self, path: str | Path) -> None:
        save_dir = Path(path)
        save_dir.mkdir(parents=True, exist_ok=True)
        self.backend.save(str(save_dir / "tokenizer.json"))
        (save_dir / "tokenizer_config.json").write_text(
            json.dumps(
                {
                    "type": "bytelevel_bpe",
                    "max_len": self.max_len,
                    "special_tokens": SPECIAL_TOKENS,
                },
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )


In [6]:
class TextDataset(Dataset):
    """Датасет для классификации текста."""

    def __init__(self, texts, labels, tokenizer, desc: str):
        self.encodings = [
            torch.tensor(tokenizer.encode(text), dtype=torch.long)
            for text in tqdm(texts, desc=desc, unit="text", leave=False)
        ]
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encodings[idx], self.labels[idx]


def collate_batch(batch):
    sequences, labels = zip(*batch)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded, torch.stack(labels)


## 8. Позиционное кодирование

In [7]:
class PositionalEncoding(nn.Module):
    """Синусоидальное позиционное кодирование."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


## 9. Архитектура трансформерного детектора

Модель состоит из следующих частей:

- эмбеддинги токенов;
- обучаемый `CLS`-токен;
- позиционное кодирование;
- стек слоёв `TransformerEncoder`;
- комбинированный pooling;
- линейная голова для бинарной классификации.

In [8]:
class AITextDetector(nn.Module):
    """Трансформерный детектор AI-текста."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        max_len: int = 512,
        dropout: float = 0.1,
        pad_idx: int = 0,
        unk_idx: int = 1,
        token_dropout: float = 0.05,
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.unk_idx = unk_idx
        self.token_dropout = token_dropout

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model, max_len + 1, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

        pooled_dim = d_model * 3
        self.head = nn.Sequential(
            nn.Linear(pooled_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    def _make_padding_mask(self, input_ids: torch.Tensor) -> torch.Tensor:
        cls_mask = torch.zeros(input_ids.size(0), 1, dtype=torch.bool, device=input_ids.device)
        pad_mask = input_ids == self.pad_idx
        return torch.cat([cls_mask, pad_mask], dim=1)

    def _apply_token_dropout(self, input_ids: torch.Tensor) -> torch.Tensor:
        if not self.training or self.token_dropout <= 0:
            return input_ids
        mask = torch.rand_like(input_ids, dtype=torch.float32) < self.token_dropout
        mask &= input_ids != self.pad_idx
        dropped = input_ids.clone()
        dropped[mask] = self.unk_idx
        return dropped

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size = input_ids.size(0)
        input_ids = self._apply_token_dropout(input_ids)
        x = self.embedding(input_ids) * math.sqrt(self.d_model)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.pos_encoder(x)

        padding_mask = self._make_padding_mask(input_ids)
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        x = self.layer_norm(x)

        cls_repr = x[:, 0]
        token_repr = x[:, 1:]
        token_mask = ~padding_mask[:, 1:]
        token_mask_f = token_mask.unsqueeze(-1).to(token_repr.dtype)

        denom = token_mask_f.sum(dim=1).clamp_min(1.0)
        mean_repr = (token_repr * token_mask_f).sum(dim=1) / denom

        masked_tokens = token_repr.masked_fill(~token_mask.unsqueeze(-1), float("-inf"))
        max_repr = masked_tokens.max(dim=1).values
        max_repr = torch.where(torch.isfinite(max_repr), max_repr, torch.zeros_like(max_repr))

        pooled = torch.cat([cls_repr, mean_repr, max_repr], dim=-1)
        logits = self.head(pooled).squeeze(-1)
        return logits


## 10. Конфигурация обучения

Гиперпараметры обучения, архитектуры и сохранения артефактов собраны в один dataclass.

In [9]:
@dataclass
class TrainConfig:
    d_model: int = 256
    nhead: int = 8
    num_layers: int = 4
    dim_feedforward: int = 512
    max_len: int = 512
    dropout: float = 0.1
    epochs: int = 10
    batch_size: int = 32
    lr: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 100
    max_vocab: int = 50_000
    val_split: float = 0.1
    seed: int = 42
    use_amp: bool = True
    token_dropout: float = 0.05
    label_smoothing: float = 0.02
    patience: int = 3
    grad_clip_val: float = 1.0
    accumulate_grad_batches: int = 1
    num_workers: int = -1
    save_dir: str = "checkpoints/student_transformer_detector"


train_config = TrainConfig(save_dir=str(artifact_dir))
print(train_config)


TrainConfig(d_model=256, nhead=8, num_layers=4, dim_feedforward=512, max_len=512, dropout=0.1, epochs=10, batch_size=32, lr=0.0003, weight_decay=0.01, warmup_steps=100, max_vocab=50000, val_split=0.1, seed=42, use_amp=True, token_dropout=0.05, label_smoothing=0.02, patience=3, grad_clip_val=1.0, accumulate_grad_batches=1, num_workers=-1, save_dir='checkpoints/student_transformer_detector')


## 11. Функция потерь и вычисление порога

Для обучения используется `BCEWithLogitsLoss` с мягким сглаживанием меток.

In [10]:
class SmoothedBCEWithLogitsLoss(nn.Module):
    def __init__(self, smoothing: float = 0.0):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.smoothing > 0:
            targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets)


def compute_optimal_threshold(logits: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    probs = 1 / (1 + np.exp(-logits))
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx = int(np.argmax(f1_scores))
    threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
    preds = (probs >= threshold).astype(int)
    return threshold, float(f1_score(labels, preds, average="macro"))


def calibrate_threshold(model: AITextDetector, loader: DataLoader, use_amp: bool) -> float:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()
    logits_list = []
    labels_list = []

    with torch.no_grad():
        for input_ids, targets in tqdm(loader, desc="Калибровка порога", unit="batch", leave=False):
            input_ids = input_ids.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp and device.type != "cpu"):
                logits = model(input_ids)
            logits_list.extend(logits.cpu().tolist())
            labels_list.extend(targets.int().cpu().tolist())

    logits_arr = np.array(logits_list, dtype=np.float32)
    labels_arr = np.array(labels_list, dtype=np.int64)
    threshold, _ = compute_optimal_threshold(logits_arr, labels_arr)
    return threshold


## 12. LightningModule

In [11]:
class AIDetectorModule(pl.LightningModule):
    def __init__(self, model: AITextDetector, cfg: TrainConfig, total_steps: int):
        super().__init__()
        self.model = model
        self.cfg = cfg
        self.total_steps = total_steps
        self.criterion = SmoothedBCEWithLogitsLoss(cfg.label_smoothing)
        self.validation_logits = []
        self.validation_targets = []

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.model(input_ids)

    def training_step(self, batch, batch_idx):
        input_ids, targets = batch
        logits = self(input_ids)
        loss = self.criterion(logits, targets)
        acc = ((logits > 0).long() == targets.long()).float().mean()
        self.log("train_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        self.log("train_acc", acc, on_epoch=True, on_step=False, prog_bar=True)
        return {"loss": loss}

    def on_validation_epoch_start(self):
        self.validation_logits.clear()
        self.validation_targets.clear()

    def validation_step(self, batch, batch_idx):
        input_ids, targets = batch
        logits = self(input_ids)
        loss = self.criterion(logits, targets)
        self.validation_logits.append(logits.detach().cpu())
        self.validation_targets.append(targets.detach().cpu())
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)

    def on_validation_epoch_end(self):
        logits = torch.cat(self.validation_logits).numpy()
        labels = torch.cat(self.validation_targets).numpy().astype(int)
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        threshold, best_f1 = compute_optimal_threshold(logits, labels)

        self.log("val_acc", float((preds == labels).mean()), prog_bar=True)
        self.log("val_f1_macro", f1_score(labels, preds, average="macro"), prog_bar=True)
        self.log("val_best_f1", best_f1)
        self.log("val_best_threshold", threshold)
        if len(np.unique(labels)) > 1:
            self.log("val_roc_auc", roc_auc_score(labels, probs))

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)
        warmup_steps = min(self.cfg.warmup_steps, max(self.total_steps - 1, 0))

        def lr_lambda(current_step: int) -> float:
            if self.total_steps <= 1:
                return 1.0
            if warmup_steps > 0 and current_step < warmup_steps:
                return 0.1 + 0.9 * (current_step / max(1, warmup_steps))
            progress = (current_step - warmup_steps) / max(1, self.total_steps - warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

        scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }


## 13. Индикатор прогресса обучения

In [12]:
class NotebookProgressBar(pl.Callback):
    def __init__(self, refresh_rate: int = 1):
        super().__init__()
        self.refresh_rate = max(1, refresh_rate)
        self.train_bar = None
        self.fit_start_time = None
        self.epoch_start_time = None

    def on_fit_start(self, trainer, pl_module):
        self.fit_start_time = time.time()

    def on_train_epoch_start(self, trainer, pl_module):
        total_batches = int(trainer.num_training_batches)
        self.epoch_start_time = time.time()
        if self.train_bar is not None:
            self.train_bar.close()
        self.train_bar = tqdm(
            total=total_batches,
            desc=f"Эпоха {trainer.current_epoch + 1}/{trainer.max_epochs}",
            leave=True,
            dynamic_ncols=True,
            unit="batch",
        )

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if self.train_bar is None:
            return
        current_batch = batch_idx + 1
        if current_batch % self.refresh_rate != 0 and current_batch != int(trainer.num_training_batches):
            return

        increment = current_batch - self.train_bar.n
        if increment > 0:
            self.train_bar.update(increment)

        elapsed_total = time.time() - self.fit_start_time if self.fit_start_time else 0.0
        elapsed_epoch = time.time() - self.epoch_start_time if self.epoch_start_time else 0.0
        total_batches = max(1, int(trainer.num_training_batches))
        eta_epoch = (elapsed_epoch / max(1, current_batch)) * max(total_batches - current_batch, 0)
        completed_epochs = trainer.current_epoch + current_batch / total_batches
        remaining_epochs = max(trainer.max_epochs - completed_epochs, 0.0)
        eta_total = (elapsed_total / completed_epochs) * remaining_epochs if completed_epochs > 0 else 0.0

        loss_value = None
        if isinstance(outputs, dict) and "loss" in outputs:
            loss_value = outputs["loss"]
        if torch.is_tensor(loss_value):
            loss_value = float(loss_value.detach().cpu())

        postfix = {
            "прошло": format_seconds(elapsed_total),
            "до конца эпохи": format_seconds(eta_epoch),
            "до конца обучения": format_seconds(eta_total),
        }
        if loss_value is not None:
            postfix["loss"] = f"{loss_value:.4f}"
        self.train_bar.set_postfix(postfix)

    def on_validation_start(self, trainer, pl_module):
        if self.train_bar is not None:
            self.train_bar.set_postfix_str("идёт валидация")

    def on_train_epoch_end(self, trainer, pl_module):
        if self.train_bar is None:
            return
        if self.train_bar.n < self.train_bar.total:
            self.train_bar.update(self.train_bar.total - self.train_bar.n)
        epoch_elapsed = time.time() - self.epoch_start_time if self.epoch_start_time else 0.0
        self.train_bar.set_postfix({"время эпохи": format_seconds(epoch_elapsed)})
        self.train_bar.close()
        self.train_bar = None

    def on_fit_end(self, trainer, pl_module):
        total_elapsed = time.time() - self.fit_start_time if self.fit_start_time else 0.0
        print(f"Общее время обучения: {format_seconds(total_elapsed)}")


## 14. Обёртка для инференса и структуры результатов

In [13]:
@dataclass
class DetectionResult:
    label: str
    confidence: float
    logit: float


@dataclass
class EvalResult:
    accuracy: float
    f1: float
    f1_human: float
    f1_ai: float
    precision_human: float
    precision_ai: float
    recall_human: float
    recall_ai: float
    roc_auc: float
    confusion: np.ndarray
    threshold_used: float
    optimal_threshold: float
    report: str


@dataclass
class SourceBreakdown:
    source: str
    n_samples: int
    accuracy: float
    f1: float
    roc_auc: float


In [14]:
class NotebookDetector:
    def __init__(
        self,
        model: AITextDetector,
        tokenizer: NotebookBPETokenizer,
        device: str = "auto",
        threshold: float = 0.5,
        use_amp: bool = True,
    ):
        self.device = torch.device(
            device if device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.model = model.to(self.device).eval()
        self.tokenizer = tokenizer
        self.threshold = threshold
        self.use_amp = use_amp

    @torch.no_grad()
    def predict(self, text: str) -> DetectionResult:
        ids = torch.tensor([self.tokenizer.encode(text)], dtype=torch.long, device=self.device)
        with torch.autocast(device_type=self.device.type, enabled=self.use_amp and self.device.type != "cpu"):
            logit = self.model(ids).item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
        is_ai = prob >= self.threshold
        return DetectionResult(
            label="AI" if is_ai else "Human",
            confidence=prob if is_ai else 1 - prob,
            logit=logit,
        )


## 15. Функции оценки качества

In [15]:
def evaluate_model_local(
    model: AITextDetector,
    tokenizer: NotebookBPETokenizer,
    texts,
    labels,
    batch_size: int = 32,
    device: str = "auto",
    threshold: float = 0.5,
    use_amp: bool = True,
) -> EvalResult:
    dev = torch.device(device if device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu"))
    model = model.to(dev).eval()

    dataset = TextDataset(texts, labels, tokenizer, desc="Кодирование теста")
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_batch, shuffle=False)

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for input_ids, targets in tqdm(loader, desc="Оценка на тесте", unit="batch", leave=False):
            input_ids = input_ids.to(dev)
            with torch.autocast(device_type=dev.type, enabled=use_amp and dev.type != "cpu"):
                logits = model(input_ids)
            all_logits.extend(logits.cpu().tolist())
            all_labels.extend(targets.int().tolist())

    logits_arr = np.array(all_logits)
    labels_arr = np.array(all_labels)
    probs = 1 / (1 + np.exp(-logits_arr))
    precision_vals, recall_vals, thresholds_pr = precision_recall_curve(labels_arr, probs)
    f1_vals = 2 * precision_vals * recall_vals / (precision_vals + recall_vals + 1e-8)
    best_idx = int(np.argmax(f1_vals))
    optimal_threshold = float(thresholds_pr[best_idx]) if best_idx < len(thresholds_pr) else 0.5
    preds = (probs >= threshold).astype(int)

    report = classification_report(labels_arr, preds, target_names=["Human", "AI"], digits=4, zero_division=0)
    cm = confusion_matrix(labels_arr, preds)
    f1_per_class = f1_score(labels_arr, preds, average=None)
    precision_per_class = precision_score(labels_arr, preds, average=None, zero_division=0)
    recall_per_class = recall_score(labels_arr, preds, average=None, zero_division=0)

    auc = roc_auc_score(labels_arr, probs) if len(np.unique(labels_arr)) > 1 else 0.0

    return EvalResult(
        accuracy=accuracy_score(labels_arr, preds),
        f1=f1_score(labels_arr, preds, average="macro"),
        f1_human=float(f1_per_class[0]),
        f1_ai=float(f1_per_class[1]) if len(f1_per_class) > 1 else 0.0,
        precision_human=float(precision_per_class[0]),
        precision_ai=float(precision_per_class[1]) if len(precision_per_class) > 1 else 0.0,
        recall_human=float(recall_per_class[0]),
        recall_ai=float(recall_per_class[1]) if len(recall_per_class) > 1 else 0.0,
        roc_auc=float(auc),
        confusion=cm,
        threshold_used=threshold,
        optimal_threshold=optimal_threshold,
        report=report,
    )


def evaluate_per_source_local(detector: NotebookDetector, samples: list[Sample]) -> list[SourceBreakdown]:
    grouped = defaultdict(lambda: ([], []))
    for sample in samples:
        texts, labels = grouped[sample.source_dataset]
        texts.append(sample.text)
        labels.append(sample.label.value if hasattr(sample.label, "value") else int(sample.label))

    results = []
    for source, (texts, labels) in tqdm(sorted(grouped.items()), desc="Анализ по источникам", unit="source", leave=False):
        if len(set(labels)) < 2:
            continue
        preds = []
        probs = []
        for text in texts:
            prediction = detector.predict(text)
            preds.append(1 if prediction.label == "AI" else 0)
            probs.append(prediction.confidence if prediction.label == "AI" else 1 - prediction.confidence)

        labels_arr = np.array(labels)
        preds_arr = np.array(preds)
        probs_arr = np.array(probs)
        auc = roc_auc_score(labels_arr, probs_arr) if len(np.unique(labels_arr)) > 1 else 0.0
        results.append(
            SourceBreakdown(
                source=source,
                n_samples=len(texts),
                accuracy=accuracy_score(labels_arr, preds_arr),
                f1=f1_score(labels_arr, preds_arr, average="macro"),
                roc_auc=float(auc),
            )
        )

    return results


In [16]:
def print_eval_summary(result: EvalResult):
    print("\nИтоговая оценка на тестовой выборке")
    print("-" * 60)
    print(f"Accuracy:      {result.accuracy:.4f}")
    print(f"Macro F1:      {result.f1:.4f}")
    print(f"ROC-AUC:       {result.roc_auc:.4f}")
    print(f"Порог:         {result.threshold_used:.3f}")
    print(f"Лучший порог:  {result.optimal_threshold:.3f}")
    print()
    print(f"Precision Human: {result.precision_human:.4f}")
    print(f"Recall Human:    {result.recall_human:.4f}")
    print(f"F1 Human:        {result.f1_human:.4f}")
    print(f"Precision AI:    {result.precision_ai:.4f}")
    print(f"Recall AI:       {result.recall_ai:.4f}")
    print(f"F1 AI:           {result.f1_ai:.4f}")
    print()
    print("Матрица ошибок:")
    print(result.confusion)
    print()
    print("Полный classification report:")
    print(result.report)


def print_source_summary(breakdowns: list[SourceBreakdown]):
    print("\nМетрики по источникам")
    print("-" * 60)
    for item in breakdowns:
        print(
            f"{item.source:20s} | сэмплов={item.n_samples:6d} | "
            f"accuracy={item.accuracy:.4f} | f1={item.f1:.4f} | roc_auc={item.roc_auc:.4f}"
        )


def label_to_russian(label: str) -> str:
    return "AI" if label == "AI" else "Человек"


## 16. Сохранение минимального набора артефактов

In [17]:
def save_artifacts(
    model: AITextDetector,
    tokenizer: NotebookBPETokenizer,
    cfg: TrainConfig,
    threshold: float,
):
    save_dir = Path(cfg.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), save_dir / "best_model.pt")
    tokenizer.save(save_dir / "tokenizer")
    model_config = {
        "vocab_size": tokenizer.vocab_size,
        "d_model": cfg.d_model,
        "nhead": cfg.nhead,
        "num_layers": cfg.num_layers,
        "dim_feedforward": cfg.dim_feedforward,
        "max_len": cfg.max_len,
        "dropout": cfg.dropout,
        "pad_idx": tokenizer.pad_idx,
        "unk_idx": tokenizer.unk_idx,
        "token_dropout": 0.0,
    }
    (save_dir / "model_config.json").write_text(
        json.dumps(model_config, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    (save_dir / "inference_config.json").write_text(
        json.dumps({"threshold": threshold}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


## 17. Полный цикл обучения

In [18]:
def train_pipeline(
    texts: list[str],
    labels: list[int],
    cfg: TrainConfig,
) -> tuple[AITextDetector, NotebookBPETokenizer, float]:
    torch.manual_seed(cfg.seed)
    configure_torch_runtime()
    num_workers = resolve_num_workers(cfg.num_workers)

    with tqdm(total=8, desc="Этапы train pipeline", unit="этап") as stage_bar:
        stage_bar.set_postfix_str("разбиение train/val")
        train_texts, val_texts, train_labels, val_labels = split_train_val(
            texts,
            labels,
            cfg.val_split,
            cfg.seed,
        )
        stage_bar.update(1)

        stage_bar.set_postfix_str("обучение токенизатора")
        tokenizer = NotebookBPETokenizer.from_texts(
            train_texts,
            max_vocab=cfg.max_vocab,
            max_len=cfg.max_len,
        )
        stage_bar.update(1)

        stage_bar.set_postfix_str("кодирование train")
        train_ds = TextDataset(train_texts, train_labels, tokenizer, desc="Кодирование train")
        stage_bar.update(1)

        stage_bar.set_postfix_str("кодирование val")
        val_ds = TextDataset(val_texts, val_labels, tokenizer, desc="Кодирование val")
        stage_bar.update(1)

        stage_bar.set_postfix_str("создание dataloader")
        train_loader = DataLoader(
            train_ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            collate_fn=collate_batch,
            num_workers=num_workers,
            persistent_workers=num_workers > 0,
            pin_memory=torch.cuda.is_available(),
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=cfg.batch_size,
            shuffle=False,
            collate_fn=collate_batch,
            num_workers=num_workers,
            persistent_workers=num_workers > 0,
            pin_memory=torch.cuda.is_available(),
        )
        print(f"Словарь: {tokenizer.vocab_size} | Train: {len(train_ds)} | Val: {len(val_ds)}")
        stage_bar.update(1)

        stage_bar.set_postfix_str("инициализация модели")
        model = AITextDetector(
            vocab_size=tokenizer.vocab_size,
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            num_layers=cfg.num_layers,
            dim_feedforward=cfg.dim_feedforward,
            max_len=cfg.max_len,
            dropout=cfg.dropout,
            pad_idx=tokenizer.pad_idx,
            unk_idx=tokenizer.unk_idx,
            token_dropout=cfg.token_dropout,
        )
        n_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
        print(f"Количество обучаемых параметров: {n_params:,}")
        stage_bar.update(1)

        stage_bar.set_postfix_str("обучение модели")
        steps_per_epoch = max(1, math.ceil(len(train_loader) / max(cfg.accumulate_grad_batches, 1)))
        total_steps = cfg.epochs * steps_per_epoch
        lit_model = AIDetectorModule(model, cfg, total_steps)

        checkpoint_cb = ModelCheckpoint(
            dirpath=str(Path(cfg.save_dir) / "lightning_checkpoints"),
            filename="best_checkpoint",
            monitor="val_f1_macro",
            mode="max",
            save_top_k=1,
        )
        early_stopping_cb = EarlyStopping(
            monitor="val_f1_macro",
            mode="max",
            patience=cfg.patience,
        )
        progress_cb = NotebookProgressBar(refresh_rate=1)

        print(f"Эпох: {cfg.epochs} | train-батчей на эпоху: {len(train_loader)} | val-батчей: {len(val_loader)}")
        print(f"Оценка числа optimizer steps: {total_steps}")

        use_amp = cfg.use_amp and torch.cuda.is_available()
        trainer = pl.Trainer(
            max_epochs=cfg.epochs,
            callbacks=[checkpoint_cb, early_stopping_cb, progress_cb],
            precision="16-mixed" if use_amp else "32-true",
            enable_progress_bar=False,
            logger=False,
            gradient_clip_val=cfg.grad_clip_val,
            accumulate_grad_batches=cfg.accumulate_grad_batches,
            enable_model_summary=False,
            default_root_dir=cfg.save_dir,
        )
        trainer.fit(lit_model, train_loader, val_loader)

        best_checkpoint = torch.load(checkpoint_cb.best_model_path, map_location="cpu", weights_only=True)
        state_dict = {
            key.removeprefix("model."): value
            for key, value in best_checkpoint["state_dict"].items()
            if key.startswith("model.")
        }
        model.load_state_dict(state_dict)
        stage_bar.update(1)

        stage_bar.set_postfix_str("калибровка и сохранение")
        threshold = calibrate_threshold(model, val_loader, use_amp)
        save_artifacts(model, tokenizer, cfg, threshold)
        print(f"Откалиброванный порог: {threshold:.3f}")
        stage_bar.update(1)

    return model, tokenizer, threshold


## 18. Загрузка объединённого датасета

На этом этапе загружается общий корпус текстов из разных источников, а затем строится отложенная тестовая выборка.

In [19]:
result = load_combined_dataset(data_config)
train_texts, train_labels, test_texts, test_labels, test_samples = split_loaded_data(
    result,
    test_ratio=test_ratio,
    seed=train_config.seed,
)

print(f"Всего примеров: {len(result.texts):,}")
print(f"Примеров для обучения: {len(train_texts):,}")
print(f"Примеров для теста: {len(test_texts):,}")
print("Распределение меток в train:", Counter(train_labels))
print("Распределение меток в test:", Counter(test_labels))
print("Источники в test:", Counter(sample.source_dataset for sample in test_samples))


Skipping ruatd because it duplicates coat.

Skipping daigt_v2 because it duplicates daigt_proper.

/Users/ep/Documents/Masters/project/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Loaded raid: 200 samples (human=200, ai=0)

Loaded ai_pile: 99,852 samples (human=49,996, ai=49,856)

...

Combined dataset: 200,000 samples

Human: 100,038 | AI: 99,962

Sources: ['raid', 'ai_pile', 'gpt_wiki', 'human_vs_ai', 'ai_human_mixed', 'hc3', 'coat', 'gsingh_train']

Generators: {'human': 100038, 'ai': 39319, 'chatgpt': 21168, 'ruGPT3-Large': 9705, 'ruGPT3-Small': 8075, 
'ruGPT3-Medium': 7959, 'OPUS-MT': 2965, 'M2M-100': 2950, 'M-BART50': 2802, 'ruGPT2-Large': 1468, 'mT5-Small': 1442,
'ruT5-Base-Multitask': 1396, 'mT5-Large': 570, 'gpt-3': 35, 'GPT_4-o': 19, 'mistral-7B': 17, 'qwen-2-72B': 17, 
'llama-8B': 16, 'gemma-2-9b': 14, 'accounts/yi-01-ai/models/yi-large': 13, 'gpt4/bard': 12}

Всего примеров: 200,000
Примеров для обучения: 170,000
Примеров для теста: 30,000
Распределение меток в train: Counter({0: 85032, 1: 84968})
Распределение меток в test: Counter({0: 15006, 1: 14994})
Источники в test: Counter({'ai_pile': 10623, 'coat': 10606, 'hc3': 8718, 'raid': 20, 'gsingh_train': 16, 'gpt_wiki': 10, 'human_vs_ai': 4, 'ai_human_mixed': 3})


## 19. Обучение модели

Ниже запускается основной эксперимент: обучение трансформерного детектора на объединённом наборе данных.

In [20]:
model, tokenizer, threshold = train_pipeline(train_texts, train_labels, train_config)


Этапы train pipeline:  62%|██████▎   | 5/8 [01:08<00:56, 18.80s/этап, инициализация модели] /var/folders/wl/k0p20njs69dddbhjrqzb6szr0000gn/T/ipykernel_50360/1437848980.py:36: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Этапы train pipeline:  75%|███████▌  | 6/8 [01:09<00:17,  8.89s/этап, обучение модели]     

Словарь: 50000 | Train: 153000 | Val: 17000
Количество обучаемых параметров: 15,106,305
Эпох: 10 | train-батчей на эпоху: 4782 | val-батчей: 532
Оценка числа optimizer steps: 47820


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/ep/Documents/Masters/project/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=90, pipe_handle=232)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ep/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/e

BrokenPipeError: [Errno 32] Broken pipe

## 20. Оценка на тестовой выборке

Теперь модель оценивается на данных, которые не использовались при обучении.

In [ ]:
eval_result = evaluate_model_local(
    model,
    tokenizer,
    test_texts,
    test_labels,
    batch_size=train_config.batch_size,
    threshold=threshold,
    use_amp=train_config.use_amp,
)
print_eval_summary(eval_result)


## 21. Анализ качества по источникам

Такой анализ помогает понять, одинаково ли хорошо модель работает на разных подкорпусах и разных стилях данных.

In [ ]:
detector = NotebookDetector(model, tokenizer, threshold=threshold, use_amp=train_config.use_amp)
breakdowns = evaluate_per_source_local(detector, test_samples)
print_source_summary(breakdowns)


## 22. Примеры предсказаний

Ниже показана работа модели на нескольких коротких текстах. Это позволяет увидеть не только средние метрики, но и поведение модели на конкретных примерах.

In [ ]:
correct = 0
for text, expected in live_examples:
    prediction = detector.predict(text)
    ok = prediction.label == expected
    correct += int(ok)
    print(
        {
            "совпадение": ok,
            "ожидалось": label_to_russian(expected),
            "предсказано": label_to_russian(prediction.label),
            "уверенность": round(prediction.confidence, 4),
            "текст": text,
        }
    )

print(f"Точность на примерах: {correct}/{len(live_examples)}")
print(f"Артефакты сохранены в: {artifact_dir}")
print(f"Используемый порог: {threshold:.3f}")


## 23. Выводы

В данной работе был реализован детектор AI-текста на основе трансформерного энкодера.

Основные результаты работы:

- реализована самодостаточная архитектура трансформерного детектора внутри ноутбука;
- использована BPE-токенизация и обучение на объединённом многодоменном корпусе;
- обучение организовано через PyTorch Lightning;
- оценка проведена как на общей тестовой выборке, так и по отдельным источникам;
- показана работа модели на отдельных примерах.

Главный вывод состоит в том, что трансформер является сильной базовой архитектурой для задачи различения human- и AI-текстов, так как способен учитывать контекст, структуру и стилистические особенности последовательности.